# Mapping between technology fields and IPC (`TLS901_TECHN_FIELD_IPC`)

The table `TLS901_TECHN_FIELD_IPC` is a reference table that provides a mapping between 35 technology fields and the more detailed IPC classification.  
These technology fields are the same as those used by the EPO and WIPO for their official statistics and are designed to group patent applications according to their technological domain.  
The content of this table is sourced from the WIPO Intellectual Property Statistics, specifically from the IPC concordance table.

Let’s break down the key fields in this table. The first step is to initialise the PATSTAT client and import the table.

In [1]:
from epo.tipdata.patstat import PatstatClient

# Initialise the PATSTAT client
patstat = PatstatClient(env='PROD')

# Access ORM
db = patstat.orm()

# Importing the models
from epo.tipdata.patstat.database.models import TLS901_TECHN_FIELD_IPC
# Importing sql alchemy package
from sqlalchemy import func

## Key fields in the `TLS901_TECHN_FIELD_IPC` table
* `IPC_MAINGROUP_SYMBOL` (primary key): this attribute stores the IPC subclass (i.e. the first four characters) or the IPC main group of an IPC symbol, as defined by WIPO Standard ST.3. It enables the table to be joined to any other table containing IPC or CPC-based classification fields.
* `TECHN_FIELD`: this attribute identifies the name of the technology field.
* `TECHN_SECTOR`: this attribute identifies the technology sector to which the field belongs. The 35 technology fields are grouped into five technology sectors.
* `TECHN_FIELD_NR`: this attribute uniquely identifies a technology field through a number.

To determine the technical fields corresponding to each application ID, the table ``TLS901_TECHN_FIELD_IPC`` can be joined with ``TLS209_APPLN_IPC`` using ``IPC_MAINGROUP_SYMBOL`` as the linking key.

In [3]:
#Import the two tables TLS901_TECHN_FIELD_IPC and TLS209_APPLN_IPC 
from epo.tipdata.patstat.database.models import TLS901_TECHN_FIELD_IPC, TLS209_APPLN_IPC 

#Join the two table using the IPC_MAINGROUP_SYMBOL in order to retrieve the technological fields associated with each application id
tech_id = (
    db.query(
        TLS209_APPLN_IPC.appln_id,
        TLS901_TECHN_FIELD_IPC.ipc_maingroup_symbol,
        TLS901_TECHN_FIELD_IPC.techn_field,
        TLS901_TECHN_FIELD_IPC.techn_sector,
        TLS901_TECHN_FIELD_IPC.techn_field_nr,
    )

# NB: The attribute `ipc_class_symbol` can have up to fourteen characters, whereas `ipc_maingroup_symbol` has only four. For this reason, when merging the two tables,it is important to consider only the first four characters of both attributes (identifing the IPC subclasses) by truncating `ipc_class_symbol`.
# To do this, we use the `substring` function from SQLAlchemy’s `func` module.

    .join(
        TLS901_TECHN_FIELD_IPC,
        TLS901_TECHN_FIELD_IPC.ipc_maingroup_symbol== func.substr(TLS209_APPLN_IPC.ipc_class_symbol,1,4)  
    )
        .order_by(
           TLS209_APPLN_IPC.appln_id #Order by the application id
        )
        .distinct()  #Add " distinct" to remove duplicate
    .limit(10000)
)

#Show the results as a dataframe
tech_df= patstat.df(tech_id)
tech_df

,appln_id,ipc_maingroup_symbol,techn_field,techn_sector,techn_field_nr
0,1,H04W,Digital communication,Electrical engineering,4
1,1,G06K,Computer technology,Electrical engineering,6
2,1,H01R,"Electrical machinery, apparatus, energy",Electrical engineering,1
3,1,H04M,Telecommunications,Electrical engineering,3
4,2,C07K,Biotechnology,Chemistry,15
...,...,...,...,...,...
9995,7897,C09K,Basic materials chemistry,Chemistry,19
9996,7899,G06F,Computer technology,Electrical engineering,6
9997,7901,G06F,Computer technology,Electrical engineering,6
9998,7903,H04W,Digital communication,Electrical engineering,4


As it is possible to see from the outcome, each application can be associated to several technological fields.

Finally, we visualise the different technical fields recorded in the table.

In [4]:
tech_fields = (
    db.query(
        TLS901_TECHN_FIELD_IPC.techn_field_nr,    
        TLS901_TECHN_FIELD_IPC.techn_field,
        TLS901_TECHN_FIELD_IPC.techn_sector,
    )
    .order_by(
        TLS901_TECHN_FIELD_IPC.techn_field_nr
    )
    .distinct()
)

#Show the result as a dataframe
tech_fields_df = patstat.df(tech_fields)
tech_fields_df

,techn_field_nr,techn_field,techn_sector
0,1,"Electrical machinery, apparatus, energy",Electrical engineering
1,2,Audio-visual technology,Electrical engineering
2,3,Telecommunications,Electrical engineering
3,4,Digital communication,Electrical engineering
4,5,Basic communication processes,Electrical engineering
5,6,Computer technology,Electrical engineering
6,7,IT methods for management,Electrical engineering
7,8,Semiconductors,Electrical engineering
8,9,Optics,Instruments
9,10,Measurement,Instruments
